# MexQL on Google Colab

Runs the stage-1 pipeline (baseline evaluation and train sampling) on a Colab GPU.
Select a GPU runtime first (Runtime > Change runtime type). A free T4 is enough for the dev
baseline (fp16); L4 or A100 (Colab Pro) is recommended for sampling the whole train set.

Outputs are small (predictions, metrics, caches) and are copied to Google Drive when `USE_DRIVE` is set.

In [ ]:
#@title Parameters
REPO_URL = "https://github.com/jakubtomczykdev/MexQL.git"  #@param {type:"string"}
MODEL = "configs/models/qwen3_4b_instruct.yaml"  #@param ["configs/models/qwen3_4b_instruct.yaml", "configs/models/qwen25_coder_3b.yaml"]
WITH_TRAIN = False  #@param {type:"boolean"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/mexql"  #@param {type:"string"}
import os
os.environ["REPO_URL"] = REPO_URL
os.environ["WITH_TRAIN"] = "1" if WITH_TRAIN else "0"


In [ ]:
#@title GPU and disk check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!df -h /content | tail -1
!python --version

In [ ]:
#@title Clone and bootstrap (uv, Python 3.11, vLLM, BIRD dev set)
%cd /content
import os, subprocess
if not os.path.isdir("MexQL"):
    subprocess.run(["git", "clone", os.environ["REPO_URL"], "MexQL"], check=True)
%cd /content/MexQL
!bash scripts/colab_bootstrap.sh

In [ ]:
#@title Optional: mount Google Drive and restore caches from a previous session
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil, pathlib
    src = pathlib.Path(DRIVE_DIR) / "processed"
    if src.is_dir():
        shutil.copytree(src, "data/processed", dirs_exist_ok=True)
        print("restored", src)

In [ ]:
#@title Prepare caches and the prompt-length report
!.venv/bin/python scripts/prepare_data.py --model $MODEL --sources dev
# add 'train' to --sources once the train set is downloaded (WITH_TRAIN)

In [ ]:
#@title Baseline: greedy decoding on dev (chunked and resumable; outputs mirrored to Drive after every chunk)
SYNC = f"--sync-dir {DRIVE_DIR}/outputs" if USE_DRIVE else ""
!.venv/bin/python scripts/evaluate.py --model $MODEL --split dev --chunk-size 200 --resume $SYNC
!cat outputs/runs/*_dev/metrics.md

In [ ]:
#@title Baseline: greedy decoding on val_internal (needs the train databases)
if WITH_TRAIN:
    !.venv/bin/python scripts/evaluate.py --model $MODEL --split val_internal --chunk-size 100 --resume $SYNC
else:
    print("skipped: set WITH_TRAIN to download the train databases")

In [ ]:
#@title Stage-2 preparation: 8 samples per question (first 200 train questions; --limit 0 = whole train_rl split)
if WITH_TRAIN:
    !.venv/bin/python scripts/sample_train.py --model $MODEL --limit 200 --chunk-size 100 --resume $SYNC
    !cat outputs/sampling/*/summary.json
else:
    print("skipped: set WITH_TRAIN to download the train databases")

In [ ]:
#@title Save outputs and caches to Drive
if USE_DRIVE:
    import shutil, pathlib
    dst = pathlib.Path(DRIVE_DIR)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree("outputs", dst / "outputs", dirs_exist_ok=True)
    shutil.copytree("data/processed", dst / "processed", dirs_exist_ok=True)
    print("saved to", dst)